# 01 — Data Exploration

Explore the labelled pause events from CANDOR/AMI:
- Label distribution & class balance
- Gap duration distributions by label
- Speaker-level stats
- Audio waveform samples for each class

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import soundfile as sf
import IPython.display as ipd

sns.set_theme(style='darkgrid', palette='muted')
FIGURES = Path('../notebooks/figures')
FIGURES.mkdir(exist_ok=True)

In [ ]:
df = pd.read_parquet('../data/processed/labels.parquet')
print(f"Total samples: {len(df):,}")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Label distribution
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#81c784', '#e57373'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
for bar, v in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{v:,}', ha='center', fontsize=10)

# Gap duration distribution by label
for label, color in [('turn_end', '#e57373'), ('mid_thought', '#81c784')]:
    subset = df[df['label'] == label]['gap_ms'].clip(upper=2000)
    axes[1].hist(subset, bins=40, alpha=0.6, label=label, color=color)
axes[1].set_title('Pause Gap Duration by Label')
axes[1].set_xlabel('Gap duration (ms)')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].axvline(700, color='white', linestyle='--', alpha=0.5, label='700ms threshold')

fig.suptitle('Cadence — Pause Event Dataset', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / 'data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Gap duration stats (ms):')
df.groupby('label')['gap_ms'].describe().round(1)

In [ ]:
# Listen to random samples of each class
SAMPLE_RATE = 16000
WINDOW_SAMPLES = 2 * SAMPLE_RATE

def load_window(row):
    start = max(0, int((row['pause_start_ms'] - 2000) * SAMPLE_RATE / 1000))
    audio, sr = sf.read(row['audio_path'], start=start, stop=start + WINDOW_SAMPLES, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    import librosa
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    return audio

for label in ['turn_end', 'mid_thought']:
    sample = df[df['label'] == label].sample(1).iloc[0]
    audio = load_window(sample)
    print(f"\n--- {label} (gap={sample['gap_ms']:.0f}ms) ---")
    display(ipd.Audio(audio, rate=SAMPLE_RATE))

In [ ]:
# Sessions and speaker counts
print(f"Unique sessions:  {df['session_id'].nunique()}")
print(f"Unique speakers:  {df['speaker_id'].nunique()}")
print(f"Corpora:          {df['corpus'].value_counts().to_dict()}")